# LSTM + Greedy

## 사용방법
- 이 노트북에서는 3개의 언어(num, en, ko)에 대해 양방향 번역을 할 수 있는 모델 하나를 학습하고 사용합니다.
- torch는 운영환경에 맞게 따로 설치 후, ```pip install tqdm notebook```로 필요한 라이브러리를 설치해주세요.
  - 작성자는 GPU 환경에서 다음과 같이 torch를 설치했습니다. (참고: https://pytorch.org/get-started/locally/)
    ```
    pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu132
    ```
- 노트북의 코드를 실행하기 전에 이 파일과 동일한 경로에 .pt 파일과 .txt 파일이 있는지 확인하세요.
- 트레이닝 코드(5번째 셀)만 주석처리했습니다. 주석을 유지한 채로 8번 항목 코드까지 실행 시 테스트 데이터로 번역 성공률을 확인할 수 있습니다.
- 직접 모델 트레이닝 시 첫 번째 셀의 MODEL_FILE 값을 변경 후 run_epoch 함수가 작성된 5번째 셀의 주석을 해제 후 실행해주세요.

# 1. 설정

In [1]:
import random, time, re
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from tqdm import tqdm

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE, EMB_DIM, HIDDEN_DIM, EPOCHS = 128, 128, 256, 30
MODEL_FILE = '1_LSTM_multi.pt'
print('Device:', DEVICE)

Device: cuda


# 2. 데이터 확인 및 6방향 데이터쌍 만들기

In [2]:

def load_file(path):
    with open(path, encoding='utf-8') as f:
        return dict(line.rstrip('\n').split('\t', 1)
                    for line in f if line.strip())

en = load_file('num2txt.txt')
ko = load_file('num2kor.txt')

LANGS = ('num', 'en', 'ko')

def tokenize(text, lang):
    return text.split() if lang == 'en' else list(text)

# 숫자별로 존재하는 언어 데이터를 모두 보관
records = {}

for num, english in en.items():
    records.setdefault(num, {'num': num})['en'] = english

for num, korean in ko.items():
    records.setdefault(num, {'num': num})['ko'] = korean

rows = list(records.values())

train_rows, val_rows, test_rows = random_split(
    rows,
    [0.8, 0.1, 0.1],
    generator=torch.Generator().manual_seed(SEED)
)

def make_pairs(data):
    pairs = []
    for texts in data:
        for src in LANGS:
            for tgt in LANGS:
                if src != tgt and src in texts and tgt in texts:
                    pairs.append((src, tgt, texts[src], texts[tgt]))
    return pairs

train_pairs = make_pairs(train_rows)
val_pairs = make_pairs(val_rows)
test_pairs = make_pairs(test_rows)

special = ['<pad>', '<bos>', '<eos>', '<unk>'] + [f'<2{x}>' for x in LANGS]  # 하나의 모델로 6방향 학습 위함
tokens = set()

for src, tgt, a, b in train_pairs:
    tokens.update(tokenize(a, src))
    tokens.update(tokenize(b, tgt))

vocab = special + sorted(tokens - set(special))
stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for token, i in stoi.items()}

PAD, BOS, EOS, UNK = (stoi[x] for x in special[:4])

print('English records:', len(en))
print('Korean records:', len(ko))
print('Unique numbers:', len(rows))
print('Vocabulary:', len(vocab))
print('Training pairs:', len(train_pairs))

English records: 100000
Korean records: 100000
Unique numbers: 187262
Vocabulary: 65
Training pairs: 340204


# 3. 텍스트 쌍을 텐서 배치로 변경

In [3]:
def ids(text, lang):
    return [stoi.get(token, UNK) for token in tokenize(text, lang)]

class TranslationDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        src, tgt, a, b = self.pairs[index]
        x = [stoi[f'<2{tgt}>']] + ids(a, src) + [EOS]  # 목표언어 알려주기
        y = [BOS] + ids(b, tgt) + [EOS]
        return torch.tensor(x), torch.tensor(y)

def collate(batch):
    x, y = zip(*batch)
    return (pad_sequence(x, batch_first=True, padding_value=PAD),
            pad_sequence(y, batch_first=True, padding_value=PAD))

train_loader = DataLoader(TranslationDataset(train_pairs), batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=collate)
val_loader = DataLoader(TranslationDataset(val_pairs), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=collate)

# 4. LSTM 모델 정의

In [4]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMB_DIM, padding_idx=PAD)
        self.encoder = nn.LSTM(EMB_DIM, HIDDEN_DIM, batch_first=True)
        self.decoder = nn.LSTM(EMB_DIM, HIDDEN_DIM, batch_first=True)
        self.fc = nn.Linear(HIDDEN_DIM, vocab_size)

    def encode(self, x):
        lengths = (x != PAD).sum(1).cpu()
        embedded = self.embedding(x)
        packed = pack_padded_sequence(embedded, lengths, batch_first=True,
                                      enforce_sorted=False)
        _, state = self.encoder(packed)
        return state

    def forward(self, x, y):
        state = self.encode(x)
        out, _ = self.decoder(self.embedding(y[:, :-1]), state)  # 정답 시퀀스를 디코더에 입력
        return self.fc(out)

model = Seq2Seq(len(vocab)).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
print('Parameters:', sum(p.numel() for p in model.parameters()))

Parameters: 815553


# 5. 학습

In [ ]:
# def run_epoch(loader, training):
#     model.train(training)
#     total_loss, total_tokens = 0.0, 0
#     for x, y in tqdm(loader, leave=False):
#         x, y = x.to(DEVICE), y.to(DEVICE)
#         with torch.set_grad_enabled(training):
#             logits = model(x, y)
#             target = y[:, 1:]
#             loss = criterion(logits.reshape(-1, len(vocab)), target.reshape(-1))
#             if training:
#                 optimizer.zero_grad()
#                 loss.backward()
#                 nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#                 optimizer.step()
#         count = (target != PAD).sum().item()
#         total_loss += loss.item() * count
#         total_tokens += count
#     return total_loss / total_tokens

# best_loss = float('inf')
# for epoch in range(1, EPOCHS + 1):
#     start = time.time()
#     train_loss = run_epoch(train_loader, True)
#     val_loss = run_epoch(val_loader, False)
#     print(f'Epoch {epoch:02d} | train {train_loss:.4f} | val {val_loss:.4f} | {time.time()-start:.0f}s')
#     if val_loss < best_loss:
#         best_loss = val_loss
#         torch.save({'model': model.state_dict(), 'vocab': vocab,
#                     'emb_dim': EMB_DIM, 'hidden_dim': HIDDEN_DIM}, MODEL_FILE)
# print('Best validation loss:', round(best_loss, 4))

Epoch 01 | train 0.4032 | val 0.0551 | 39s


Epoch 02 | train 0.0326 | val 0.0190 | 85s


Epoch 03 | train 0.0148 | val 0.0148 | 38s


Epoch 04 | train 0.0089 | val 0.0068 | 100s


Epoch 05 | train 0.0060 | val 0.0052 | 121s


Epoch 06 | train 0.0043 | val 0.0055 | 114s


... (20 outputs left out)

Epoch 27 | train 0.0005 | val 0.0010 | 34s


Epoch 28 | train 0.0005 | val 0.0011 | 37s


Epoch 29 | train 0.0004 | val 0.0006 | 37s


Epoch 30 | train 0.0004 | val 0.0006 | 34s
Best validation loss: 0.0006


# 6. 디코딩 알고리즘 정의: Greedy

In [6]:
@torch.inference_mode()
def translate(text, source_lang, target_lang, net=model, max_len=60):
    net.eval()
    x = [stoi[f'<2{target_lang}>']] + ids(str(text), source_lang) + [EOS]
    x = torch.tensor([x], device=DEVICE)
    state = net.encode(x)
    token = torch.tensor([[BOS]], device=DEVICE)
    output = []
    for _ in range(max_len):
        hidden, state = net.decoder(net.embedding(token), state)
        token = net.fc(hidden).argmax(-1)  # greedy: 매 단계 가장 높은 점수 토큰 선택
        token_id = token.item()
        if token_id == EOS:
            break
        output.append(itos[token_id])
    return ' '.join(output) if target_lang == 'en' else ''.join(output)

model.load_state_dict(
    torch.load(MODEL_FILE, map_location=DEVICE, weights_only=True)['model']
)

print(f'모델 로드 완료: {MODEL_FILE}')

모델 로드 완료: 1_LSTM_multi.pt


# 7. 테스트 (방향별 하나씩)

In [7]:
for src in LANGS:
    for tgt in LANGS:
        if src == tgt:
            continue
        sample = next((p for p in test_pairs if p[0] == src and p[1] == tgt), None)
        if sample is None:
            print(f'{src}->{tgt} | 테스트 데이터 없음')
            continue
        _, _, text, gt = sample
        predicted = translate(text, src, tgt)
        correct = predicted == gt  # 예측 결과와 gt가 같으면 PASS만 표시, 다르면 결과 표시
        print(f'{src}->{tgt} | {"PASS" if correct else "FAIL"}')
        if not correct:
            print(f'  input: {text}')
            print(f'  predicted: {predicted}')
            print(f'  gt:  {gt}')

num->en | PASS
num->ko | PASS
en->num | PASS
en->ko | PASS
ko->num | PASS
ko->en | PASS


# 8. 테스트 (테스트셋 전체)

In [8]:
counts = {(a, b): [0, 0] for a in LANGS for b in LANGS if a != b}

for src, tgt, text, gt in tqdm(test_pairs):
    predicted = translate(text, src, tgt)
    counts[(src, tgt)][0] += (predicted == gt)
    counts[(src, tgt)][1] += 1

for direction, (correct, total) in counts.items():
    if total > 0:
        print(f'{direction[0]} -> {direction[1]}: {100 * correct / total:.2f}%')
    else:
        print(f'{direction[0]} -> {direction[1]}: No test data')

100%|██████████| 42492/42492 [02:58<00:00, 237.68it/s]

num -> en: 99.87%
num -> ko: 99.82%
en -> num: 99.93%
en -> ko: 99.44%
ko -> num: 99.95%
ko -> en: 99.68%


## 9. 직접 입력 테스트

- 아래는 테스트셋이 아닌 사용자 입력으로 모델의 일반화 능력을 검증합니다.
- 작성자 테스트 시, 숫자에 대한 번역은 잘하였으나 영어<->한국어 일반화 능력이 부족함을 확인했습니다.

In [9]:
LANGS_LABEL = {'num': '숫자', 'en': '영어', 'ko': '한국어'}

def fmt_num(s):
    digits = s.replace(',', '').replace(' ', '')
    if digits.isascii() and digits.isdecimal():
        return re.sub(r'\B(?=(\d{3})+(?!\d))', ',', digits)
    return s

def show(text, lang):
    return fmt_num(text) if lang == 'num' else text

def detect_lang(text):
    t = text.strip().replace(',', '')
    if t.isascii() and t.isdecimal():
        return 'num'
    return 'en' if t.isascii() else 'ko'

def demo_input(text, src):
    text = text.strip()
    if src == 'num':
        text = text.replace(',', '')
        if not text or not text.isascii() or not text.isdecimal():
            raise ValueError('숫자는 0~9만 입력해주세요. 예: 12345')
    elif src == 'en':
        text = ' '.join(text.lower().split())
        if not text:
            raise ValueError('영어 숫자 표현을 입력해주세요.')
    else:
        if not text:
            raise ValueError('한국어 숫자 표현을 입력해주세요.')
    return text

def demo_translate(text, source_lang, target_lang='all'):
    text = demo_input(text, source_lang)
    targets = ([lang for lang in LANGS if lang != source_lang]
               if target_lang == 'all' else [target_lang])

    unknown = sorted(set(token for token in tokenize(text, source_lang)
                         if token not in stoi))
    if unknown:
        print(f'주의: 학습 어휘에 없는 토큰 {unknown} → <unk> 처리됩니다.')

    print(f'\n입력 ({LANGS_LABEL[source_lang]}): {show(text, source_lang)}')
    for tgt in targets:
        result = translate(text, source_lang, tgt)
        print(f'출력 ({LANGS_LABEL[tgt]}): {show(result, tgt)}')
    print('-' * 45)

while True:
    text = input('\n입력 (종료: q): ').strip()
    if text.lower() == 'q':
        print('테스트를 종료합니다.')
        break
    if not text:
        continue
    try:
        demo_translate(text, detect_lang(text), 'all')
    except ValueError as error:
        print(f'입력 오류: {error}')


입력 (숫자): 12,345
출력 (영어): twelve thousand three hundred forty five
출력 (한국어): 만 이천삼백사십오
---------------------------------------------

입력 (한국어): 만 이천삼백사십오
출력 (숫자): 12,345
출력 (영어): twelve thousand three hundred forty five
---------------------------------------------

입력 (영어): twelve thousand three hundred forty five
출력 (숫자): 12,345
출력 (한국어): 만 이천삼백사십오
---------------------------------------------

입력 (숫자): 12,345,678
출력 (영어): twelve million three hundred forty five thousand six hundred seventy eight
출력 (한국어): 천이백삼십사만 오천육백칠십팔
---------------------------------------------

입력 (한국어): 천이백삼십사만 오천육백칠십팔
출력 (숫자): 12,345,678
출력 (영어): 십 million two hundred thirty four thousand five hundred seventy eight
---------------------------------------------

입력 (영어): twelve million three hundred forty five thousand six hundred seventy eight
출력 (숫자): 12,345,678
출력 (한국어): 이백삼십오만 사천육백칠십팔
---------------------------------------------
테스트를 종료합니다.


# 10. 추론 개선 시도
## (1) Pivot Translation
*참고: https://aclanthology.org/Q17-1024.pdf

In [10]:
def pivot_translate(text):
    source = detect_lang(text)
    text = demo_input(text, source)
    if source == 'num':
        return source, {tgt: translate(text, source, tgt)
                        for tgt in LANGS if tgt != source}

    # 영어 또는 한국어 → 숫자 → 반대 언어
    number = translate(text, source, 'num')
    if not number.isascii() or not number.isdecimal():
        raise ValueError(f'중간 숫자 변환 실패: {number}')
    target = 'ko' if source == 'en' else 'en'
    return source, {'num': number,
                    target: translate(number, 'num', target)}


# 직접 번역 vs Pivot Translation
text = 'thirty thousand'
source = detect_lang(text)
_, pivot = pivot_translate(text)
target = 'ko' if source == 'en' else 'en'
print('입력:', text)
print('직접 번역:', translate(demo_input(text, source), source, target))
print('중간 숫자:', pivot['num'])
print('Pivot 번역:', pivot[target])

입력: thirty thousand
직접 번역: 삼만 0
중간 숫자: 30000
Pivot 번역: 삼만 


## (2) Beam as Decoding Algorithm: topk

In [11]:
import torch.nn.functional as F

@torch.inference_mode()
def translate_beam(text, source_lang, target_lang, beam_width=3, net=model, max_len=60):
    if beam_width < 1 or beam_width > len(vocab):
        raise ValueError('beam_width는 1 이상, vocabulary 크기 이하여야 합니다.')
    net.eval()
    x = [stoi[f'<2{target_lang}>']] + ids(str(text), source_lang) + [EOS]
    state = net.encode(torch.tensor([x], device=DEVICE))

    # (토큰 목록, 누적 로그확률, LSTM 상태, 종료 여부)
    beams = [([BOS], 0.0, state, False)]
    for _ in range(max_len):
        candidates = []
        for tokens, score, hidden_state, finished in beams:
            if finished:
                candidates.append((tokens, score, hidden_state, True))
                continue
            token = torch.tensor([[tokens[-1]]], device=DEVICE)
            out, new_state = net.decoder(net.embedding(token), hidden_state)
            log_probs = F.log_softmax(net.fc(out)[:, -1, :], dim=-1)
            values, indices = torch.topk(log_probs, beam_width, dim=-1)
            for log_prob, token_id in zip(values[0].tolist(), indices[0].tolist()):
                candidates.append((tokens + [token_id], score + log_prob,
                                   new_state, token_id == EOS))
        beams = sorted(candidates, key=lambda b: b[1], reverse=True)[:beam_width]
        if all(b[3] for b in beams):
            break

    # Greedy와 동일하게 최상위 경로를 사용 (beam_width=1이면 Greedy와 동일)
    tokens, _, _, _ = beams[0]
    output_ids = tokens[1:]
    if output_ids and output_ids[-1] == EOS:
        output_ids = output_ids[:-1]
    return (' ' if target_lang == 'en' else '').join(itos[i] for i in output_ids)


# Greedy vs Beam Search 비교
text = 'thirty thousand'
source = detect_lang(text)
text = demo_input(text, source)
for target in LANGS:
    if target == source:
        continue
    print(f'\n{source} → {target}')
    print('Greedy:', translate(text, source, target))
    print('Beam  :', translate_beam(text, source, target, beam_width=3))


en → num
Greedy: 30000
Beam  : 30000

en → ko
Greedy: 삼만 0
Beam  : 삼만 0
